# Project 3

---
## Setup

In [1]:
import subprocess, sys

def _ensure(pkg, import_name=None):
    import importlib.util
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_ensure('requests')
_ensure('psycopg2-binary', 'psycopg2')
_ensure('kafka-python-ng', 'kafka')
print('packages ready')

packages ready


In [2]:
import os, json, time
import requests
import psycopg2
from kafka import KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient

CONNECT_URL = os.environ.get('CONNECT_URL', 'http://connect:8083')
BOOTSTRAP   = os.environ.get('KAFKA_BOOTSTRAP', 'kafka:9092')
PG = dict(
    host=os.environ.get('PG_HOST', 'postgres'),
    port=int(os.environ.get('PG_PORT', 5432)),
    dbname=os.environ.get('PG_DB', 'sourcedb'),
    user=os.environ.get('PG_USER', 'cdc_user'),
    password=os.environ.get('PG_PASSWORD', 'cdc_pass'),
)

def pg(sql, fetch=False):
    conn = psycopg2.connect(**PG); conn.autocommit = True
    cur = conn.cursor(); cur.execute(sql)
    rows = cur.fetchall() if fetch else None
    cur.close(); conn.close()
    return rows

def topic_offset(topic):
    c = KafkaConsumer(bootstrap_servers=BOOTSTRAP)
    tp = TopicPartition(topic, 0); c.assign([tp]); c.seek_to_end(tp)
    end = c.position(tp); c.close()
    return end

print('helpers ready')

helpers ready


---
## Part 1 - Service health + Postgres source

In [3]:
checks = {
    'Kafka Connect': requests.get(f'{CONNECT_URL}/').json(),
    'Iceberg REST':  requests.get('http://iceberg-rest:8181/v1/config').json(),
    'MinIO':         requests.get('http://minio:9000/minio/health/live').status_code,
    'Postgres':      pg('SHOW wal_level;', fetch=True)[0][0],
    'Airflow':       requests.get('http://airflow:8080/health').json(),
}
print(json.dumps(checks, indent=2))

{
  "Kafka Connect": {
    "version": "3.9.0",
    "commit": "a60e31147e6b01ee",
    "kafka_cluster_id": "MkU3OEVBNTcwNTJENDM2Qg"
  },
  "Iceberg REST": {
    "defaults": {},
    "overrides": {}
  },
  "MinIO": 200,
  "Postgres": "logical",
  "Airflow": {
    "dag_processor": {
      "latest_dag_processor_heartbeat": null,
      "status": null
    },
    "metadatabase": {
      "status": "healthy"
    },
    "scheduler": {
      "latest_scheduler_heartbeat": "2026-04-27T13:43:08.819018+00:00",
      "status": "healthy"
    },
    "triggerer": {
      "latest_triggerer_heartbeat": "2026-04-27T13:43:08.761016+00:00",
      "status": "healthy"
    }
  }
}


### 1.1 Seed Postgres (10 customers, 8 drivers)

In [4]:
result = subprocess.run([sys.executable, '/home/jovyan/project/seed.py'],
                        capture_output=True, text=True, check=True)
print(result.stdout)

Seeding PostgreSQL source database

wal_level = logical

Creating tables...
  ✓ customers
  ✓ drivers

Inserting customers...
  ✓ 10 customers inserted

Inserting drivers...
  ✓ 8 drivers inserted

Seed complete!

Customers:
  (1, 'Alice Mets', 'alice@example.com', 'Estonia')
  (2, 'Bob Virtanen', 'bob@example.com', 'Finland')
  (3, 'Carol Ozols', 'carol@example.com', 'Latvia')
  (4, 'David Jonaitis', 'david@example.com', 'Lithuania')
  (5, 'Eva Svensson', 'eva@example.com', 'Sweden')
  (6, 'Frank Muller', 'frank@example.com', 'Germany')
  (7, 'Grace Kim', 'grace@example.com', 'South Korea')
  (8, 'Hiro Tanaka', 'hiro@example.com', 'Japan')
  (9, 'Ingrid Larsen', 'ingrid@example.com', 'Norway')
  (10, 'Javier Garcia', 'javier@example.com', 'Spain')

Drivers:
  (1, 'Tom Driver', 'TLC-10001', Decimal('4.85'), 'Manhattan')
  (2, 'Sarah Wheels', 'TLC-10002', Decimal('4.72'), 'Brooklyn')
  (3, 'Mike Road', 'TLC-10003', Decimal('4.91'), 'Queens')
  (4, 'Lisa Lane', 'TLC-10004', Decimal('4.68

In [5]:
print('customers count:', pg('SELECT COUNT(*) FROM customers;', fetch=True)[0][0])
print('drivers count:', pg('SELECT COUNT(*) FROM drivers;',   fetch=True)[0][0])
print('first 3 customers:')
print(pg('SELECT id, name, country FROM customers ORDER BY id LIMIT 3;', fetch=True))

customers count: 10
drivers count: 8
first 3 customers:
[(1, 'Alice Mets', 'Estonia'), (2, 'Bob Virtanen', 'Finland'), (3, 'Carol Ozols', 'Latvia')]


---
## Part 2 - Debezium connector

Register the connector via the Connect REST API. Initial snapshot emits `op='r'` events for every existing row; live INSERT/UPDATE/DELETE follow as `c`/`u`/`d`.

In [6]:
requests.delete(f"{CONNECT_URL}/connectors/pg-cdc-connector")
time.sleep(2)

connector_config = {
    "name": "pg-cdc-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": "cdc_user",
        "database.password": "cdc_pass",
        "database.dbname": "sourcedb",
        "topic.prefix": "dbserver1",
        "table.include.list": "public.customers,public.drivers",
        "plugin.name": "pgoutput",
        "snapshot.mode": "initial",
        "key.converter.schemas.enable": "false",
        "value.converter.schemas.enable": "false",
        # Postgres NUMERIC/DECIMAL would otherwise be emitted as base64-encoded bytes;
        # `double` makes them arrive as JSON numbers so Spark can cast them directly.
        "decimal.handling.mode": "double",
    }
}

r = requests.post(
    f"{CONNECT_URL}/connectors",
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config),
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 201
{
  "name": "pg-cdc-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "cdc_user",
    "database.password": "cdc_pass",
    "database.dbname": "sourcedb",
    "topic.prefix": "dbserver1",
    "table.include.list": "public.customers,public.drivers",
    "plugin.name": "pgoutput",
    "snapshot.mode": "initial",
    "key.converter.schemas.enable": "false",
    "value.converter.schemas.enable": "false",
    "decimal.handling.mode": "double",
    "name": "pg-cdc-connector"
  },
  "tasks": [],
  "type": "source"
}


In [7]:
# Poll status - Connect needs a few seconds after POST to provision the task
for i in range(20):
    r = requests.get(f'{CONNECT_URL}/connectors/pg-cdc-connector/status')
    if r.status_code == 200:
        break
    time.sleep(1)

status = r.json()
print(json.dumps(status, indent=2))
print(f"connector RUNNING -> {status['connector']['state'] == 'RUNNING'}")
print(f"all tasks RUNNING -> {all(t['state'] == 'RUNNING' for t in status['tasks'])}")

{
  "name": "pg-cdc-connector",
  "connector": {
    "state": "RUNNING",
    "worker_id": "172.18.0.7:8083"
  },
  "tasks": [
    {
      "id": 0,
      "state": "RUNNING",
      "worker_id": "172.18.0.7:8083"
    }
  ],
  "type": "source"
}
connector RUNNING -> True
all tasks RUNNING -> True


### 2.1 CDC topics + per-topic event counts

Expected after a fresh seed: customers = 10, drivers = 8.

In [8]:
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
topics = sorted(t for t in admin.list_topics() if t.startswith('dbserver1'))
admin.close()
for topic in topics:
    print(f'{topic}: end offset = {topic_offset(topic)}')

dbserver1.public.customers: end offset = 10
dbserver1.public.drivers: end offset = 8


### 2.2 Inspect the first CDC event (Debezium envelope)

Confirms the envelope shape: `op`, `after.*`, `source.lsn`, `ts_ms` under `$.payload.*`.

In [9]:
c = KafkaConsumer(bootstrap_servers=BOOTSTRAP, auto_offset_reset='earliest', enable_auto_commit=False, consumer_timeout_ms=3000)
c.assign([TopicPartition('dbserver1.public.customers', 0)])
c.seek_to_beginning()
msg = next(c); c.close()
p = json.loads(msg.value)['payload']
print(f"op= {p['op']!r}")
print(f"after.id= {p['after']['id']}")
print(f"after.name= {p['after']['name']!r}")
print(f"source.lsn= {p['source']['lsn']}")
print(f"ts_ms= {p['ts_ms']}")
print(f"op == 'r' (initial snapshot read) -> {p['op'] == 'r'}")
# print(json.dumps(p,indent=2))

op= 'r'
after.id= 1
after.name= 'Alice Mets'
source.lsn= 26809856
ts_ms= 1777297393124
op == 'r' (initial snapshot read) -> True


### 2.3 Force one live UPDATE - should add one Kafka event

In [10]:
before = topic_offset('dbserver1.public.customers')
pg("UPDATE customers SET email='test@example.com' WHERE id=1;")
time.sleep(2)
after = topic_offset('dbserver1.public.customers')
print(f'before = {before}, after = {after}, delta = {after - before}')

before = 10, after = 11, delta = 1


### 2.4 Simulator burst - 5 random ops, deterministic exit

A DELETE emits two Kafka events (d-envelope + tombstone), so 5 random ops grow the topic offsets by 5-10.

In [11]:
before_c = topic_offset('dbserver1.public.customers')
before_d = topic_offset('dbserver1.public.drivers')
result = subprocess.run([sys.executable, '/home/jovyan/project/simulate.py', '--rate', '5', '--limit', '5'],
                        capture_output=True, text=True, check=True)
print(result.stdout)
time.sleep(2)
after_c = topic_offset('dbserver1.public.customers')
after_d = topic_offset('dbserver1.public.drivers')
print(f'customers: {before_c} -> {after_c}  (+{after_c - before_c})')
print(f'drivers: {before_d} -> {after_d}  (+{after_d - before_d})')

CDC Change Simulator
Rate:   5.0 ops/sec
Tables: both
Limit:  5
Press Ctrl-C to stop.

[    1 |    0.0s] DELETE customer id=5
[    2 |    0.2s] INSERT customer: Yuki Ozols (Brazil)
[    3 |    0.5s] UPDATE customer id=3: country → Italy
[    4 |    0.7s] INSERT customer: Maria Larsen (Germany)
[    5 |    0.9s] INSERT customer: Ivan Park (France)

Reached limit of 5 operations.

Stopped — 5 operations in 1s (5.0 ops/s)

Final state:
  customers: 12 rows
  drivers:   8 rows

customers: 11 -> 17  (+6)
drivers: 8 -> 8  (+0)


---
## Part 3 - Bronze CDC layer (Iceberg, append-only)

The Jupyter kernel auto-loads PySpark + the Kafka/Iceberg jars from `PYSPARK_SUBMIT_ARGS`. We import the project modules and re-use `get_spark()` and `bronze_cdc.run_one()`.

In [12]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3://warehouse/"

spark = (
    SparkSession.builder
    .appName("bdm_project3_walkthrough")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg + REST catalog ────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          S3_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Make the project's `jobs/` modules importable so we can re-use bronze/silver run_one()
sys.path.insert(0, '/home/jovyan/project/jobs')
import bronze_cdc

print(f"Spark {spark.version} catalog: lakehouse")
print(f"S3: {S3_ENDPOINT}")

Spark 4.1.0 catalog: lakehouse
S3: http://minio:9000


### 3.1 First bronze run - ingests every Kafka event so far

In [13]:
for k in ('customers', 'drivers'):
    new, total = bronze_cdc.run_one(spark, k)
    print(f'{k}: +{new} new, total = {total}')

[customers] reading from offset 0  (topic=dbserver1.public.customers)
[customers] new events to ingest: 16
[customers] bronze total rows: 16
customers: +16 new, total = 16
[drivers] reading from offset 0  (topic=dbserver1.public.drivers)
[drivers] new events to ingest: 8
[drivers] bronze total rows: 8
drivers: +8 new, total = 8


### 3.2 Idempotency - re-running with no new Kafka events appends 0 rows

In [14]:
for k in ('customers', 'drivers'):
    new, total = bronze_cdc.run_one(spark, k)
    print(f'  {k}: +{new} new, total = {total}')

[customers] reading from offset 17  (topic=dbserver1.public.customers)
[customers] new events to ingest: 0
[customers] bronze total rows: 16
  customers: +0 new, total = 16
[drivers] reading from offset 8  (topic=dbserver1.public.drivers)
[drivers] new events to ingest: 0
[drivers] bronze total rows: 8
  drivers: +0 new, total = 8


### 3.3 One live UPDATE -> one new bronze row

In [15]:
pg("UPDATE customers SET email='bronze-test@example.com' WHERE id=2;")
time.sleep(2)
new, total = bronze_cdc.run_one(spark, 'customers')
print(f'customers: +{new} new, total = {total}')

[customers] reading from offset 17  (topic=dbserver1.public.customers)
[customers] new events to ingest: 1
[customers] bronze total rows: 17
customers: +1 new, total = 17


### 3.4 All 4 op codes (`r`/`u`/`c`/`d`) land + tombstones are filtered

INSERT + DELETE produces 3 Kafka events (c-envelope + d-envelope + tombstone) but bronze grows by only 2.

In [16]:
before_kafka  = topic_offset('dbserver1.public.customers')
before_bronze = spark.table('lakehouse.cdc.bronze_customers').count()
pg("INSERT INTO customers (name, email, country) VALUES ('Test Insert', 'test-c@example.com', 'France');")
pg("DELETE FROM customers WHERE id=3;")
time.sleep(2)
after_kafka = topic_offset('dbserver1.public.customers')
new, after_bronze = bronze_cdc.run_one(spark, 'customers')
print(f'kafka: {before_kafka} -> {after_kafka}  (+{after_kafka - before_kafka})')
print(f'bronze: {before_bronze} -> {after_bronze}  (+{after_bronze - before_bronze})')
print(f'tombstones filtered: {(after_kafka - before_kafka) - (after_bronze - before_bronze)}')

[customers] reading from offset 18  (topic=dbserver1.public.customers)
[customers] new events to ingest: 2
[customers] bronze total rows: 19
kafka: 18 -> 21  (+3)
bronze: 17 -> 19  (+2)
tombstones filtered: 1


### 3.5 Schema spot-check - all 4 op codes visible

In [17]:
spark.sql("""
    SELECT op, after_id, after_name, after_email, source_lsn, ts_ms
    FROM lakehouse.cdc.bronze_customers
    ORDER BY kafka_offset
""").show(20, truncate=False)
ops = sorted({row['op'] for row in spark.sql('SELECT DISTINCT op FROM lakehouse.cdc.bronze_customers').collect()})
print(f'distinct op codes: {ops}')

+---+--------+--------------+-----------------------+----------+-------------+
|op |after_id|after_name    |after_email            |source_lsn|ts_ms        |
+---+--------+--------------+-----------------------+----------+-------------+
|r  |1       |Alice Mets    |alice@example.com      |26809856  |1777297393124|
|r  |2       |Bob Virtanen  |bob@example.com        |26809856  |1777297393126|
|r  |3       |Carol Ozols   |carol@example.com      |26809856  |1777297393126|
|r  |4       |David Jonaitis|david@example.com      |26809856  |1777297393126|
|r  |5       |Eva Svensson  |eva@example.com        |26809856  |1777297393126|
|r  |6       |Frank Muller  |frank@example.com      |26809856  |1777297393127|
|r  |7       |Grace Kim     |grace@example.com      |26809856  |1777297393127|
|r  |8       |Hiro Tanaka   |hiro@example.com       |26809856  |1777297393127|
|r  |9       |Ingrid Larsen |ingrid@example.com     |26809856  |1777297393127|
|r  |10      |Javier Garcia |javier@example.com     

### 3.6 Bronze table schemas

`DESCRIBE TABLE` for the REPORT.md §2 *"schema of each table"* requirement. Both tables share the Kafka metadata + envelope columns; `bronze_drivers` adds typed source columns including `rating DOUBLE`.

In [18]:
spark.sql("DESCRIBE TABLE lakehouse.cdc.bronze_customers").show(50, truncate=False)
spark.sql("DESCRIBE TABLE lakehouse.cdc.bronze_drivers").show(50, truncate=False)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|kafka_offset   |bigint   |NULL   |
|kafka_partition|int      |NULL   |
|kafka_timestamp|timestamp|NULL   |
|op             |string   |NULL   |
|before_id      |int      |NULL   |
|after_id       |int      |NULL   |
|before_name    |string   |NULL   |
|after_name     |string   |NULL   |
|before_email   |string   |NULL   |
|after_email    |string   |NULL   |
|before_country |string   |NULL   |
|after_country  |string   |NULL   |
|source_lsn     |bigint   |NULL   |
|ts_ms          |bigint   |NULL   |
+---------------+---------+-------+

+---------------------+---------+-------+
|col_name             |data_type|comment|
+---------------------+---------+-------+
|kafka_offset         |bigint   |NULL   |
|kafka_partition      |int      |NULL   |
|kafka_timestamp      |timestamp|NULL   |
|op                   |string   |NULL   |
|before_id            |int      |NULL   |
|after_id      

### 3.7 Iceberg snapshot history (free for REPORT.md §2)

In [19]:
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakehouse.cdc.bronze_customers.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|7385273827220518282|2026-04-27 13:44:11.187|append   |
|3158486742855632696|2026-04-27 13:44:20.104|append   |
|2635129267396745248|2026-04-27 13:44:24.781|append   |
+-------------------+-----------------------+---------+



---
## Part 4 - Silver CDC layer (MERGE INTO + dedup)

Reads bronze, deduplicates to one event per primary key (`ROW_NUMBER` over `ts_ms DESC`), then `MERGE INTO` silver:
- `op = 'd'` → DELETE
- `op IN ('c', 'u', 'r')` → UPDATE if matched, INSERT if not

After MERGE, silver mirrors the current state of the PostgreSQL source.

In [20]:
import silver_cdc

for k in ('customers', 'drivers'):
    silver_cdc.run_one(spark, k)

[customers] silver: 0 -> 12 rows
[drivers] silver: 0 -> 8 rows


### 4.1 Silver mirrors Postgres - row counts + 3-row spot check

Compare silver row counts against the live Postgres source - if they match (within tolerance), silver is a faithful current-state mirror.

In [21]:
for table, pg_table in [('customers', 'customers'), ('drivers', 'drivers')]:
    silver_n = spark.table(f'lakehouse.cdc.silver_{table}').count()
    pg_n     = pg(f'SELECT COUNT(*) FROM {pg_table};', fetch=True)[0][0]
    print(f'{table:<10} silver={silver_n:<3} postgres={pg_n:<3} match={silver_n == pg_n}')

print('\nSilver customers (5 rows):')
spark.sql('SELECT id, name, email, country FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5').show(truncate=False)

print('Postgres customers (same 5 rows for cross-check):')
print(pg('SELECT id, name, email, country FROM customers ORDER BY id LIMIT 5;', fetch=True))

customers  silver=12  postgres=12  match=True
drivers    silver=8   postgres=8   match=True

Silver customers (5 rows):
+---+--------------+-----------------------+-----------+
|id |name          |email                  |country    |
+---+--------------+-----------------------+-----------+
|1  |Alice Mets    |test@example.com       |Estonia    |
|2  |Bob Virtanen  |bronze-test@example.com|Finland    |
|4  |David Jonaitis|david@example.com      |Lithuania  |
|6  |Frank Muller  |frank@example.com      |Germany    |
|7  |Grace Kim     |grace@example.com      |South Korea|
+---+--------------+-----------------------+-----------+

Postgres customers (same 5 rows for cross-check):
[(1, 'Alice Mets', 'test@example.com', 'Estonia'), (2, 'Bob Virtanen', 'bronze-test@example.com', 'Finland'), (4, 'David Jonaitis', 'david@example.com', 'Lithuania'), (6, 'Frank Muller', 'frank@example.com', 'Germany'), (7, 'Grace Kim', 'grace@example.com', 'South Korea')]


### 4.2 Idempotency - re-running MERGE with no new bronze events leaves silver unchanged

Running the silver job twice with no new bronze events should produce the same silver state.

In [22]:
for k in ('customers', 'drivers'):
    before, after = silver_cdc.run_one(spark, k)
    print(f'  {k}: {before} -> {after}  unchanged={before == after}')

[customers] silver: 12 -> 12 rows
  customers: 12 -> 12  unchanged=True
[drivers] silver: 8 -> 8 rows
  drivers: 8 -> 8  unchanged=True


### 4.3 Live INSERT in Postgres → bronze → silver

INSERT a new customer in Postgres. Run bronze (picks up the c-event) then silver (MERGE inserts the row).

In [23]:
pg("INSERT INTO customers (name, email, country) VALUES ('Silver Test', 'silver@example.com', 'Italy');")
time.sleep(2)
bronze_cdc.run_one(spark, 'customers')
silver_cdc.run_one(spark, 'customers')

print('\nNewest silver row (should be Silver Test):')
spark.sql("SELECT id, name, email, country FROM lakehouse.cdc.silver_customers ORDER BY id DESC LIMIT 1").show(truncate=False)

[customers] reading from offset 20  (topic=dbserver1.public.customers)
[customers] new events to ingest: 1
[customers] bronze total rows: 20
[customers] silver: 12 -> 13 rows

Newest silver row (should be Silver Test):
+---+-----------+------------------+-------+
|id |name       |email             |country|
+---+-----------+------------------+-------+
|15 |Silver Test|silver@example.com|Italy  |
+---+-----------+------------------+-------+



### 4.4 DELETE in Postgres → row absent from silver

A DELETE on the source must propagate to silver - the row should be missing after the next bronze + silver run.

In [24]:
target_id = pg('SELECT id FROM customers ORDER BY id DESC LIMIT 1;', fetch=True)[0][0]
print(f'will delete customer id={target_id}')

pg(f"DELETE FROM customers WHERE id={target_id};")
time.sleep(2)
bronze_cdc.run_one(spark, 'customers')
silver_cdc.run_one(spark, 'customers')

still_in_silver = spark.sql(f'SELECT COUNT(*) AS n FROM lakehouse.cdc.silver_customers WHERE id={target_id}').collect()[0]['n']
still_in_pg = pg(f'SELECT COUNT(*) FROM customers WHERE id={target_id};', fetch=True)[0][0]
print(f'after DELETE: silver={still_in_silver} postgres={still_in_pg}  (both should be 0)')

will delete customer id=15
[customers] reading from offset 22  (topic=dbserver1.public.customers)
[customers] new events to ingest: 1
[customers] bronze total rows: 21
[customers] silver: 13 -> 12 rows
after DELETE: silver=0 postgres=0  (both should be 0)


### 4.5 Silver table schemas

For REPORT.md §2 *"schema of each table"*. Silver is typed and compact (one row per PK), unlike append-only bronze.

In [25]:
spark.sql("DESCRIBE TABLE lakehouse.cdc.silver_customers").show(50, truncate=False)
spark.sql("DESCRIBE TABLE lakehouse.cdc.silver_drivers").show(50, truncate=False)

+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |int      |NULL   |
|name      |string   |NULL   |
|email     |string   |NULL   |
|country   |string   |NULL   |
|updated_ts|bigint   |NULL   |
+----------+---------+-------+

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|id            |int      |NULL   |
|name          |string   |NULL   |
|license_number|string   |NULL   |
|rating        |double   |NULL   |
|city          |string   |NULL   |
|active        |boolean  |NULL   |
|updated_ts    |bigint   |NULL   |
+--------------+---------+-------+



### 4.6 Iceberg snapshot history + time travel

Every MERGE produces a new Iceberg snapshot. We can query the snapshots metadata and time-travel to any earlier state.

In [26]:
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakehouse.cdc.silver_customers.snapshots
    ORDER BY committed_at
""").collect()

for s in snapshots:
    print(f"  {s['snapshot_id']}  {s['committed_at']}  {s['operation']}")

# Time travel: query silver as it was at the FIRST snapshot (before any later MERGEs)
first_snap = snapshots[0]['snapshot_id']
print(f"\nSilver customers as of FIRST snapshot ({first_snap}):")
spark.sql(f"""
    SELECT id, name, email FROM lakehouse.cdc.silver_customers VERSION AS OF {first_snap}
    ORDER BY id LIMIT 5
""").show(truncate=False)

print("Silver customers NOW (latest snapshot):")
spark.sql("SELECT id, name, email FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)

  5637423585241130040  2026-04-27 13:44:36.484000  append
  318055370828722973  2026-04-27 13:44:39.064000  overwrite
  8156884486986154894  2026-04-27 13:44:45.381000  overwrite
  5546887679276990138  2026-04-27 13:44:50.428000  overwrite

Silver customers as of FIRST snapshot (5637423585241130040):
+---+--------------+-----------------------+
|id |name          |email                  |
+---+--------------+-----------------------+
|1  |Alice Mets    |test@example.com       |
|2  |Bob Virtanen  |bronze-test@example.com|
|4  |David Jonaitis|david@example.com      |
|6  |Frank Muller  |frank@example.com      |
|7  |Grace Kim     |grace@example.com      |
+---+--------------+-----------------------+

Silver customers NOW (latest snapshot):
+---+--------------+-----------------------+
|id |name          |email                  |
+---+--------------+-----------------------+
|1  |Alice Mets    |test@example.com       |
|2  |Bob Virtanen  |bronze-test@example.com|
|4  |David Jonaitis|david@e

---
## Part 5 - Taxi pipeline (Path B)

`bronze_taxi → silver_taxi → gold_taxi` - same medallion as Project 2, ported to Airflow-triggered batch. All three layers are idempotent.

Prerequisite: the producer must have published events. The cell below sends a bounded burst.

In [27]:
result = subprocess.run(
    [sys.executable, '-u', '/home/jovyan/project/produce.py', '--rate', '50', '--limit', '50'],
    capture_output=True, text=True, check=True,
)
print(result.stdout[-500:])
print(f'taxi-trips end offset = {topic_offset("taxi-trips")}')

', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Sliced   : 50 rows (--limit 50)

Connecting to Kafka at kafka:9092 …
Topic    : taxi-trips
Rate     : 50.0 events/s
Loop     : False
Press Ctrl-C to stop.

[     1 sent |    0.3s |   3.6 ev/s]  pickup=2025-01-01 00:18:38  PU=229  DO=237  fare=$10.0

Reached limit of 50 events.

Stopped — 50 events in 1s (37.1 ev/s, 1 pass(es)).

taxi-trips end offset = 50


### 5.1 Bronze → Silver → Gold (first run + idempotent re-run)

In [28]:
import bronze_taxi, silver_taxi, gold_taxi

print('=== first run ===')
bronze_taxi.run(spark)
silver_taxi.run(spark)
gold_taxi.run(spark)

print('\n=== idempotent re-run ===')
bronze_taxi.run(spark)
silver_taxi.run(spark)
gold_taxi.run(spark)

=== first run ===
[taxi] reading from offset 0  (topic=taxi-trips)
[taxi] new events to ingest: 50
[taxi] bronze total rows: 50
[taxi-silver] reading bronze rows with kafka offset > -1
[taxi-silver] new bronze rows: 50
[taxi-silver] inserted=45, rejected=5, silver total = 45
[taxi-gold] silver rows: 45
[taxi-gold] gold aggregated rows: 44

=== idempotent re-run ===
[taxi] reading from offset 50  (topic=taxi-trips)
[taxi] new events to ingest: 0
[taxi] bronze total rows: 50
[taxi-silver] reading bronze rows with kafka offset > 49
[taxi-silver] new bronze rows: 0
[taxi-silver] silver total rows: 45
[taxi-gold] silver rows: 45
[taxi-gold] gold aggregated rows: 44


(44, 44)

### 5.2 Sample rows from each layer + schemas

In [29]:
print('--- bronze sample ---')
spark.sql("SELECT vendor_id_key, offset, kafka_timestamp, substr(json_payload, 1, 80) AS json_head FROM lakehouse.taxi.bronze ORDER BY offset LIMIT 3").show(truncate=False)

print('--- silver sample (parsed + zone-enriched) ---')
spark.sql("""
    SELECT VendorID, pickup_datetime, trip_distance, fare_amount, pickup_zone, dropoff_zone
    FROM lakehouse.taxi.silver ORDER BY pickup_datetime LIMIT 3
""").show(truncate=False)

print('--- gold sample (hourly aggregation) ---')
spark.sql("""
    SELECT hour, pickup_zone, dropoff_zone, trip_count, total_revenue, avg_fare
    FROM lakehouse.taxi.gold ORDER BY trip_count DESC LIMIT 5
""").show(truncate=False)

--- bronze sample ---
+-------------+------+-----------------------+--------------------------------------------------------------------------------+
|vendor_id_key|offset|kafka_timestamp        |json_head                                                                       |
+-------------+------+-----------------------+--------------------------------------------------------------------------------+
|1            |0     |2026-04-27 13:45:08.204|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:18:38", "tpep_dropoff_dat|
|1            |1     |2026-04-27 13:45:08.225|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:32:40", "tpep_dropoff_dat|
|1            |2     |2026-04-27 13:45:08.246|{"VendorID": 1, "tpep_pickup_datetime": "2025-01-01T00:44:04", "tpep_dropoff_dat|
+-------------+------+-----------------------+--------------------------------------------------------------------------------+

--- silver sample (parsed + zone-enriched) ---
+--------+-------------------+----

---
## Part 6 - Airflow DAG (`lakehouse`)

[`dags/lakehouse_dag.py`](dags/lakehouse_dag.py) - 7 tasks: `connector_health → [bronze_cdc, bronze_taxi] → [silver_cdc, silver_taxi] → gold_taxi → validate`. Schedule `@hourly`, `retries=3` with exponential backoff, `sla=30min`, `on_failure_callback`.

Below: register the Airflow Connection the HttpSensor needs, then trigger and poll the DAG end-to-end via the REST API.

In [30]:
AIRFLOW = 'http://airflow:8080'
AUTH = ('admin', 'admin')

# 1. Create the debezium_connect Airflow Connection (idempotent: delete then post)
requests.delete(f'{AIRFLOW}/api/v1/connections/debezium_connect', auth=AUTH)
r = requests.post(
    f'{AIRFLOW}/api/v1/connections', auth=AUTH,
    json={'connection_id': 'debezium_connect', 'conn_type': 'http', 'host': 'connect', 'port': 8083, 'schema': 'http'},
)
print(f"connection: HTTP {r.status_code}")

# 2. Unpause the DAG
r = requests.patch(f'{AIRFLOW}/api/v1/dags/lakehouse', auth=AUTH, json={'is_paused': False})
print(f"unpause: is_paused={r.json().get('is_paused')}")

connection: HTTP 200
unpause: is_paused=False


### 6.1 Trigger the DAG and poll until terminal

In [31]:
def trigger_and_poll(suffix=''):
    run_id = f"notebook__{int(time.time())}{suffix}"
    requests.post(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns', auth=AUTH,
                  json={'dag_run_id': run_id})
    print(f'triggered {run_id}')
    while True:
        s = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns/{run_id}', auth=AUTH).json()['state']
        if s in ('success', 'failed'):
            print(f'  -> {s}')
            return run_id, s
        time.sleep(10)

run_id, state = trigger_and_poll()

# Per-task report
ti = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns/{run_id}/taskInstances', auth=AUTH).json()['task_instances']
for t in sorted(ti, key=lambda x: x.get('start_date') or ''):
    print(f"  {t['task_id']:<20} {t.get('state'):<10} dur={t.get('duration')}s")

triggered notebook__1777297528
  -> success
  connector_health     success    dur=0.198578s
  bronze_taxi          success    dur=12.404631s
  bronze_cdc           success    dur=13.645256s
  silver_taxi          success    dur=13.504748s
  silver_cdc           success    dur=13.318871s
  gold_taxi            success    dur=13.866012s
  validate             success    dur=9.125324s


### 6.2 Run history (latest 5 runs of the `lakehouse` DAG)

Confirms multiple consecutive successful runs. Open the Airflow UI at http://localhost:8094 (admin/admin) for graph + grid screenshots.

In [32]:
runs = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns?order_by=-start_date&limit=5', auth=AUTH).json()['dag_runs']
for r in runs:
    print(f"  {r['dag_run_id']:<35}  state={r['state']:<10}  start={r['start_date'][:19]}")

  notebook__1777297528                 state=success     start=2026-04-27T13:47:24
  scheduled__2026-04-27T12:00:00+00:00  state=success     start=2026-04-27T13:45:28


### 6.3 Fault injection - kill connector, watch DAG block, restart, watch recover

Demonstrates failure isolation + recovery: a downstream outage blocks the DAG at the sensor (no downstream tasks run with stale state), and the DAG resumes automatically once the dependency is back.

This cell:
1. Deletes the connector.
2. Triggers a run.
3. Waits ~30 s - `connector_health` should be in `running` (HttpSensor still poking).
4. Re-registers the connector.
5. Waits for the run to finish - should succeed end-to-end once the sensor's next poke sees RUNNING.

In [33]:
# 1. Delete connector
requests.delete(f'{CONNECT_URL}/connectors/pg-cdc-connector')
print('connector deleted')

# 2. Trigger run
fail_run_id = f"notebook__fault_{int(time.time())}"
requests.post(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns', auth=AUTH, json={'dag_run_id': fail_run_id})
print(f'triggered {fail_run_id}')

# 3. Snapshot mid-run while connector is gone
time.sleep(30)
ti = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns/{fail_run_id}/taskInstances', auth=AUTH).json()['task_instances']
print('--- snapshot 1: connector still deleted ---')
for t in sorted(ti, key=lambda x: x['task_id']):
    print(f"  {t['task_id']:<20} {t.get('state')}")

# 4. Recover: re-register the connector
print('\n--- re-registering connector ---')
subprocess.check_call([sys.executable, '/home/jovyan/project/jobs/register_connector.py'])

# 5. Wait for run to finish
while True:
    s = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns/{fail_run_id}', auth=AUTH).json()['state']
    if s in ('success', 'failed'):
        break
    time.sleep(15)
print(f'\nfinal state: {s}')
ti = requests.get(f'{AIRFLOW}/api/v1/dags/lakehouse/dagRuns/{fail_run_id}/taskInstances', auth=AUTH).json()['task_instances']
print('--- snapshot 2: post-recovery ---')
for t in sorted(ti, key=lambda x: x.get('start_date') or ''):
    print(f"  {t['task_id']:<20} {t.get('state'):<10} dur={t.get('duration')}s")

connector deleted
triggered notebook__fault_1777297748
--- snapshot 1: connector still deleted ---
  bronze_cdc           None
  bronze_taxi          None
  connector_health     running
  gold_taxi            None
  silver_cdc           None
  silver_taxi          None
  validate             None

--- re-registering connector ---

final state: success
--- snapshot 2: post-recovery ---
  connector_health     success    dur=40.21903s
  bronze_taxi          success    dur=18.790259s
  bronze_cdc           success    dur=15.43393s
  silver_taxi          success    dur=13.404226s
  silver_cdc           success    dur=16.536651s
  gold_taxi            success    dur=17.265269s
  validate             success    dur=10.55158s


---
## Part 7 - Custom scenario - Customer Geography
Two new gold tables under `lakehouse.cdc`, both fed from silver+bronze CDC and orchestrated by a separate `customer_geography` Airflow DAG (file [`dags/geography_dag.py`](dags/geography_dag.py)):

- **`gold_customer_geography`** - per-country snapshot, full `INSERT OVERWRITE` each run.
- **`gold_geography_trend`** - one row per `(run_id, country)`, append-only, idempotent on `run_id`.

Spark job: [`jobs/gold_geography.py`](jobs/gold_geography.py).


### 7.1 First snapshot (`snap_1`)

`gold_geography.main()` creates both tables on first call (idempotent DDL), reads silver_customers for active counts and bronze_customers for 24h add/delete deltas, then writes the geography snapshot and appends a trend row per country.


In [34]:
import gold_geography
gold_geography.main(["--run-id", "snap_1"])


[geography] gold_customer_geography rebuilt: 12 rows for run_id=snap_1
[geography] trend appended 12 rows for run_id=snap_1
[geography] trend total rows: 12


(12, 12)

### 7.2 Geography table after `snap_1`


In [35]:
spark.sql("""
    SELECT country, active_customers, added_24h, deleted_24h, net_change_24h,
           ROUND(pct_of_total, 1) AS pct
    FROM lakehouse.cdc.gold_customer_geography
    ORDER BY active_customers DESC
""").show(truncate=False)


+-----------+----------------+---------+-----------+--------------+----+
|country    |active_customers|added_24h|deleted_24h|net_change_24h|pct |
+-----------+----------------+---------+-----------+--------------+----+
|France     |2               |2        |0          |2             |16.7|
|Germany    |2               |1        |0          |1             |16.7|
|Norway     |1               |0        |0          |0             |8.3 |
|South Korea|1               |0        |0          |0             |8.3 |
|Brazil     |1               |1        |0          |1             |8.3 |
|Finland    |1               |0        |0          |0             |8.3 |
|Japan      |1               |0        |0          |0             |8.3 |
|Spain      |1               |0        |0          |0             |8.3 |
|Estonia    |1               |0        |0          |0             |8.3 |
|Lithuania  |1               |0        |0          |0             |8.3 |
|           |0               |0        |3          

### 7.3 Two more snapshots with real movement

Each cycle: mutate Postgres -> re-run bronze_cdc + silver_cdc to propagate the change -> re-run geography with a new run_id. Using `pg()` directly (rather than the simulator) keeps the changes deterministic so the trend table is easy to read.


In [36]:
import silver_cdc

# --- Cycle 2: insert a new customer in a country that's currently absent (Italy was deleted earlier) ---
pg("INSERT INTO customers (name, email, country) VALUES ('Geo Test 1', 'geo1@example.com', 'Italy');")
time.sleep(2)
bronze_cdc.run_one(spark, 'customers')
silver_cdc.run_one(spark, 'customers')
gold_geography.main(["--run-id", "snap_2"])


[customers] reading from offset 23  (topic=dbserver1.public.customers)
[customers] new events to ingest: 1
[customers] bronze total rows: 22
[customers] silver: 12 -> 13 rows
[geography] gold_customer_geography rebuilt: 12 rows for run_id=snap_2
[geography] trend appended 12 rows for run_id=snap_2
[geography] trend total rows: 24


(12, 12)

In [37]:
# --- Cycle 3: delete that same customer so Italy drops back to 0 (-> dropped_20pct=True) ---
pg("DELETE FROM customers WHERE email='geo1@example.com';")
time.sleep(2)
bronze_cdc.run_one(spark, 'customers')
silver_cdc.run_one(spark, 'customers')
gold_geography.main(["--run-id", "snap_3"])


[customers] reading from offset 25  (topic=dbserver1.public.customers)
[customers] new events to ingest: 1
[customers] bronze total rows: 23
[customers] silver: 13 -> 12 rows
[geography] gold_customer_geography rebuilt: 12 rows for run_id=snap_3
[geography] trend appended 12 rows for run_id=snap_3
[geography] trend total rows: 36


(12, 12)

### 7.4 Demo queries

"Which country has the most customers?" - taken straight from the latest geography snapshot (geography is fully overwritten each run, so this is by construction the live answer).

"Which lost the most in the last hour?" - the most negative `delta` from trend rows whose `snapshot_ts` is within the last hour.


In [38]:
print('Most customers:')
spark.sql("""
    SELECT country, active_customers
    FROM lakehouse.cdc.gold_customer_geography
    ORDER BY active_customers DESC
    LIMIT 1
""").show()

print('Lost the most in the last hour:')
spark.sql("""
    SELECT country, prev_active, active_customers, delta,
           ROUND(pct_change, 1) AS pct_change, dropped_20pct
    FROM lakehouse.cdc.gold_geography_trend
    WHERE snapshot_ts > current_timestamp() - INTERVAL 1 HOUR
    ORDER BY delta ASC
    LIMIT 1
""").show()


Most customers:
+-------+----------------+
|country|active_customers|
+-------+----------------+
| France|               2|
+-------+----------------+

Lost the most in the last hour:
+-------+-----------+----------------+-----+----------+-------------+
|country|prev_active|active_customers|delta|pct_change|dropped_20pct|
+-------+-----------+----------------+-----+----------+-------------+
|  Italy|          1|               0|   -1|    -100.0|         true|
+-------+-----------+----------------+-----+----------+-------------+



### 7.5 Trend table - >=3 snapshots over time

One row per country per run. `prev_active` is NULL on a country's first appearance (no baseline to compute change against). `dropped_20pct` flips to True when `pct_change <= -20%`.


In [39]:
spark.sql("""
    SELECT run_id, snapshot_ts, country, active_customers, prev_active, delta,
           ROUND(pct_change, 1) AS pct_change, dropped_20pct
    FROM lakehouse.cdc.gold_geography_trend
    ORDER BY snapshot_ts, country
""").show(50, truncate=False)


+------+--------------------------+-----------+----------------+-----------+-----+----------+-------------+
|run_id|snapshot_ts               |country    |active_customers|prev_active|delta|pct_change|dropped_20pct|
+------+--------------------------+-----------+----------------+-----------+-----+----------+-------------+
|snap_1|2026-04-27 13:51:45.164199|           |0               |NULL       |0    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.164199|Brazil     |1               |NULL       |1    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.164199|Estonia    |1               |NULL       |1    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.164199|Finland    |1               |NULL       |1    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.164199|France     |2               |NULL       |2    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.164199|Germany    |2               |NULL       |2    |NULL      |false        |
|snap_1|2026-04-27 13:51:45.

### 7.6 Idempotency check

Re-run with `snap_3` again - geography overwrites with the same rows, trend skips the append (logs `skipping`). Trend total should be unchanged.


In [40]:
before = spark.table('lakehouse.cdc.gold_geography_trend').count()
gold_geography.main(["--run-id", "snap_3"])
after  = spark.table('lakehouse.cdc.gold_geography_trend').count()
print(f'trend rows before={before}  after={after}  unchanged={before == after}')


[geography] gold_customer_geography rebuilt: 12 rows for run_id=snap_3
[geography] trend already has run_id=snap_3 (12 rows), skipping append
[geography] trend total rows: 36
trend rows before=36  after=36  unchanged=True


---
## Summary

| Layer / Component | Table or File | Purpose |
|-------|-------|-------------|
| Bronze CDC | `lakehouse.cdc.bronze_customers`, `bronze_drivers` | Append-only Debezium envelope, typed cols from `$.payload.*` |
| Silver CDC | `lakehouse.cdc.silver_customers`, `silver_drivers` | MERGE INTO mirror of Postgres source |
| Bronze taxi | `lakehouse.taxi.bronze` | Append-only raw JSON + Kafka metadata |
| Silver taxi | `lakehouse.taxi.silver` | Parsed, cleaned, zone-enriched, partitioned by `pickup_date` |
| Gold taxi | `lakehouse.taxi.gold` | Hourly aggregation, `INSERT OVERWRITE` (idempotent) |
| Airflow DAG | [`dags/lakehouse_dag.py`](dags/lakehouse_dag.py) | 7-task orchestration: `connector_health → bronze_* → silver_* → gold_taxi → validate` |

See [REPORT_DELIVERABLE.md](REPORT_DELIVERABLE.md) for the full writeup with concrete numbers + the chosen Project-2 improvements.